# TSX Stock Breakout Scanner (with SL, T1, T2, T3)

## What does this code do?
- Scans 30 TSX (Toronto Stock Exchange) stocks
- Checks moving average alignment (30, 50, 200 day)
- Confirms trend strength using CAR (Cumulative Average Return)
- Automatically calculates SL (Stop Loss) and targets (T1, T2, T3)
- Displays results in a clean table format

## Step 1: Import Required Libraries

These libraries are needed to download stock data, process it, and display the results.

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import warnings
import logging

# Turn off Yahoo Finance warnings and unnecessary logs (for clean output)
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

print('All required libraries loaded successfully!')

All required libraries loaded successfully!


## Step 2: Build the Main Scanner Function

This function:
- Downloads stock data
- Calculates moving averages (30, 50, 200 day)
- Checks CAR (whether the trend is getting stronger)
- Automatically calculates SL and targets

In [ ]:
def advanced_stock_scanner(ticker_list):
    """
    Stock scanner function - finds breakout signals

    Parameters:
    - ticker_list: list of TSX stocks (e.g., ['RY.TO', 'TD.TO'])

    Returns the stocks that pass all filters
    """

    results = []  # empty list to store results
    today_date = datetime.now().strftime('%d-%m-%Y')  # today's date

    print(f'Scanning {len(ticker_list)} stocks... please wait.\n')

    for ticker in ticker_list:
        try:
            # ===================================
            # Step 1: Download stock data
            # ===================================
            # Download 2 years of daily data (260 trading days = 1 year)
            data = yf.download(ticker, period='2y', interval='1d', progress=False)

            # Check the data - skip this stock if it's empty or too short
            if data.empty or len(data) < 200:
                continue

            # ===================================
            # Step 2: Calculate moving averages
            # ===================================
            close_prices = data['Close'].squeeze()  # extract closing prices

            # Calculate 30, 50, 200 day moving averages
            dma_30 = float(close_prices.rolling(window=30).mean().iloc[-1])
            dma_50 = float(close_prices.rolling(window=50).mean().iloc[-1])
            dma_200 = float(close_prices.rolling(window=200).mean().iloc[-1])
            cmp = float(close_prices.iloc[-1])  # today's closing price (current price)

            # Calculate distance from 200-DMA in percentage
            # This shows how far above the 200-DMA the price currently is
            dist_200_dma = ((cmp - dma_200) / dma_200) * 100

            # ===================================
            # Step 3: Calculate CAR (Cumulative Average Return)
            # ===================================
            # This checks whether the trend is progressively getting stronger

            # Find the highest high in the last 252 trading days (1 year)
            last_1y_data = data.tail(252)
            high_date = last_1y_data['High'].squeeze().idxmax()

            # Take closing prices after that date
            car_data = close_prices.loc[high_date:]

            # Skip this stock if the data is too short
            if len(car_data) < 10:
                continue

            # Calculate Cumulative Average Return (expanding mean)
            car_values = car_data.expanding().mean()
            last_10_car = car_values.tail(10)  # last 10 days

            # Check whether CAR has been continuously rising over the last 10 days
            car_status = 'Positive' if last_10_car.is_monotonic_increasing else 'Negative'

            # ===================================
            # Step 4: Apply the YouTuber's filters (all 4 conditions must be met)
            # ===================================
            # Condition 1: Price > 30-DMA (above short-term trend)
            # Condition 2: Price > 50-DMA (above medium-term trend)
            # Condition 3: Price > 200-DMA (above long-term trend)
            # Condition 4: CAR is positive (trend is strengthening)

            if not ((cmp > dma_30) and (cmp > dma_50) and (cmp > dma_200) and (car_status == 'Positive')):
                continue  # this stock did not pass the filters, move to the next one

            # ===================================
            # Step 5: Calculate SL, targets and risk
            # ===================================

            # Stop Loss = lowest price of the last 10 days
            # This shows the level at which we exit if the trade goes wrong
            swing_low = float(data['Low'].tail(10).min().item())
            sl = swing_low  # stop loss = swing low

            # Entry = current price
            entry = cmp

            # Risk = entry - stop loss (how much can be lost, in dollars)
            risk = entry - sl

            # Skip this stock if risk is invalid
            if risk <= 0:
                continue

            # Calculate targets (based on risk)
            # T1 = entry + (2 x risk) - minimum target
            # T2 = entry + (3 x risk) - primary target
            # T3 = entry + (5 x risk) - aggressive target
            t1 = entry + (2 * risk)
            t2 = entry + (3 * risk)
            t3 = entry + (5 * risk)

            # ===================================
            # Step 6: Gather additional information
            # ===================================

            # Find the 52-week high (last 252 trading days)
            high_52 = float(data['High'].tail(252).max().item())

            # Check trend strength
            # If 30-DMA > 50-DMA > 200-DMA, the trend is strongly up
            if dma_30 > dma_50 > dma_200:
                trend = 'Strong Uptrend'
            else:
                trend = 'Weak'

            # ===================================
            # Step 7: Add the result to the results list
            # ===================================
            results.append({
                'Date': today_date,
                'Stock': ticker.replace('.TO', ''),
                'CMP': round(entry, 2),
                '30 DMA': round(dma_30, 2),
                '50 DMA': round(dma_50, 2),
                '200 DMA': round(dma_200, 2),
                '200 DMA Dist %': round(dist_200_dma, 2),
                'SL': round(sl, 2),
                'T1': round(t1, 2),
                'T2': round(t2, 2),
                'T3': round(t3, 2),
                'CAR Status': car_status,
                'Action': 'Positive Breakout'
            })

        except Exception as e:
            # If any error occurs, skip this stock and move to the next one
            pass

    # ===================================
    # Convert results to a DataFrame and return
    # ===================================
    if results:
        df = pd.DataFrame(results)
        # Sort by 200 DMA Dist % (lowest first)
        df = df.sort_values(by='200 DMA Dist %', ascending=True)
        return df
    else:
        return pd.DataFrame()

## Step 3: Build the Stock List and Run the Scanner

This will scan 30 major TSX (Toronto Stock Exchange) stocks

In [ ]:
# List of 30 major TSX stocks (Yahoo Finance tickers use the .TO suffix)
my_stocks = [
    'RY.TO', 'TD.TO', 'BNS.TO', 'BMO.TO', 'CM.TO',
    'ENB.TO', 'TRP.TO', 'SU.TO', 'CNQ.TO', 'CVE.TO',
    'CNR.TO', 'CP.TO', 'BCE.TO', 'T.TO', 'RCI-B.TO',
    'SHOP.TO', 'CSU.TO', 'BN.TO', 'BAM.TO', 'MFC.TO',
    'SLF.TO', 'POW.TO', 'L.TO', 'ATD.TO', 'QSR.TO',
    'ABX.TO', 'AEM.TO', 'NTR.TO', 'FNV.TO', 'WCN.TO'
]

print('\n' + '='*120)
print('TSX Stock Scanner - Breakout + SL, T1, T2, T3')
print('='*120 + '\n')

# Run the scanner and get the results
result = advanced_stock_scanner(my_stocks)


TSX Stock Scanner - Breakout + SL, T1, T2, T3

Scanning 30 stocks... please wait.



## Step 4: Display the Results

In [ ]:
# Display results if any stock passed the filters
if result.empty:
    print('\nNo stocks met all the conditions today.\n')
else:
    # Show the most important columns
    display_cols = ['Date', 'Stock', 'CMP', '30 DMA', '50 DMA', '200 DMA',
                    '200 DMA Dist %', 'SL', 'T1', 'T2', 'T3', 'CAR Status', 'Action']

    print(result[display_cols].to_string(index=False))

    print('\n' + '='*120)
    print(f'Total stocks: {len(result)}')
    print('='*120)

      Date Stock   CMP  30 DMA  50 DMA  200 DMA  200 DMA Dist %    SL     T1     T2     T3 CAR Status            Action
01-08-2026     L 65.81   64.54   64.11    62.33            5.59 63.18  71.07  73.70  78.96   Positive Positive Breakout
01-08-2026   CNQ 66.78   60.27   61.28    55.16           21.08 60.55  79.24  85.47  97.93   Positive Positive Breakout
01-08-2026    SU 94.14   84.45   85.33    75.57           24.57 86.44 109.54 117.24 132.64   Positive Positive Breakout
01-08-2026   CVE 42.27   37.99   38.36    31.48           34.27 38.52  49.77  53.52  61.02   Positive Positive Breakout

Total stocks: 4


## Step 5: Save Results to Excel

In [ ]:
# Export results to an Excel file so you can analyze them
if not result.empty:
    filename = f'TSX_Scanner_Results_{datetime.now().strftime("%d-%m-%Y_%H%M%S")}.xlsx'
    result.to_excel(filename, index=False)
    print(f'Results saved: {filename}')
else:
    print('No data to save to Excel.')

Results saved: TSX_Scanner_Results_01-08-2026_184107.xlsx


## Step 6: Understanding the Results

### Column Glossary:
- **Date**: Date of the scan
- **Stock**: Stock ticker (e.g., RY, TD, SHOP)
- **CMP**: Current market price (entry point)
- **30 DMA**: 30-day moving average
- **50 DMA**: 50-day moving average
- **200 DMA**: 200-day moving average
- **200 DMA Dist %**: How far above the 200-DMA the current price is (in %)
- **SL**: Stop loss level (last 10-day low)
- **T1**: First target (2x risk)
- **T2**: Primary target (3x risk) - take profit here
- **T3**: Aggressive target (5x risk)
- **CAR Status**: Whether the trend is strengthening
- **Action**: Buy signal (Positive Breakout)

### Trading Notes:
1. **Exit at SL** if the price breaks below the SL level
2. **Take partial profit at T2** (or T1)
3. **Trail your risk toward T3** (or hold for aggressive traders)
4. **Check Risk:Reward** - generally 1:2 or better is preferred
5. Prices are quoted in CAD (Canadian Dollars), since TSX is a Canadian exchange

## Bonus Strategy: RSI 5-Star Setup

A second, independent scan - a multi-timeframe RSI pullback strategy:
- **Monthly RSI > 60** - long-term momentum is strong
- **Weekly RSI > 60** - medium-term momentum is strong
- **Daily RSI near 40** - price has pulled back on the daily chart (the "signal candle")
- **Entry** = break above the signal candle's high
- **Stop Loss** = lowest low of the swing (last 10 days up to the signal candle)
- **1st Target** = the price level where daily RSI would reach back up to 60

> Note: many traders use a 3-5 bar trailing stop once the trade is in profit, instead of a
> fixed exit. This scanner reports the initial stop and first target only - manage the trade
> with your own trailing rules from there.

**Implementation notes** (since "near 40" isn't a single exact number):
- "Near 40" is treated as daily RSI(14) between 35 and 45
- The signal candle is the most recent day in the last 15 trading days whose RSI falls in that band
- A stock is only reported once price has actually closed above the signal candle's high (the entry trigger has already fired)
- The 1st target is estimated by solving Wilder's RSI formula for the price that would put the next daily RSI reading at 60

In [ ]:
def compute_rsi(price_series, period=14):
    """
    Calculates RSI (Relative Strength Index) using Wilder's smoothing method.
    Returns the RSI series plus the average gain/loss series (needed later
    to estimate a price target for a given RSI level).
    """
    delta = price_series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder's smoothing = an EWM with alpha = 1/period
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi_series = 100 - (100 / (1 + rs))
    return rsi_series, avg_gain, avg_loss


def rsi_five_star_scanner(ticker_list):
    """
    RSI 5-Star scanner - multi-timeframe RSI pullback strategy

    Rules:
    - Monthly RSI > 60
    - Weekly RSI > 60
    - Daily RSI near 40 (35-45) on a recent "signal candle"
    - Entry = price closes above the signal candle's high
    - Stop Loss = lowest low of the last 10 days up to the signal candle
    - Target 1 = estimated price where daily RSI would reach 60
    """
    RSI_PERIOD = 14
    SIGNAL_RSI_LOW, SIGNAL_RSI_HIGH = 35, 45
    SIGNAL_LOOKBACK_DAYS = 15
    SL_LOOKBACK_DAYS = 10
    TARGET_RSI = 60

    results = []
    today_date = datetime.now().strftime('%d-%m-%Y')

    print(f'Scanning {len(ticker_list)} stocks for the RSI 5-Star setup... please wait.\n')

    for ticker in ticker_list:
        try:
            data = yf.download(ticker, period='2y', interval='1d', progress=False)
            if data.empty or len(data) < 220:
                continue

            close = data['Close'].squeeze()
            high = data['High'].squeeze()
            low = data['Low'].squeeze()

            # Monthly and weekly RSI, from resampled closing prices
            monthly_close = close.resample('ME').last().dropna()
            weekly_close = close.resample('W').last().dropna()
            if len(monthly_close) < RSI_PERIOD + 1 or len(weekly_close) < RSI_PERIOD + 1:
                continue

            monthly_rsi, _, _ = compute_rsi(monthly_close, RSI_PERIOD)
            weekly_rsi, _, _ = compute_rsi(weekly_close, RSI_PERIOD)
            daily_rsi, daily_avg_gain, daily_avg_loss = compute_rsi(close, RSI_PERIOD)

            last_monthly_rsi = float(monthly_rsi.iloc[-1])
            last_weekly_rsi = float(weekly_rsi.iloc[-1])

            # Condition 1 & 2: Monthly and Weekly RSI both above 60
            if not (last_monthly_rsi > 60 and last_weekly_rsi > 60):
                continue

            # Condition 3: find the most recent "signal candle" - daily RSI near 40
            recent_rsi = daily_rsi.tail(SIGNAL_LOOKBACK_DAYS)
            signal_mask = (recent_rsi >= SIGNAL_RSI_LOW) & (recent_rsi <= SIGNAL_RSI_HIGH)
            if not signal_mask.any():
                continue
            signal_date = signal_mask[signal_mask].index[-1]
            signal_high = float(high.loc[signal_date])

            # Condition 4: Entry = price has broken above the signal candle's high
            cmp = float(close.iloc[-1])
            if cmp <= signal_high:
                continue

            # Stop Loss = lowest low of the swing (10 days up to the signal candle)
            swing_window = low.loc[:signal_date].tail(SL_LOOKBACK_DAYS)
            sl = float(swing_window.min())
            entry = signal_high
            risk = entry - sl
            if risk <= 0:
                continue

            # Target 1: solve Wilder's RSI formula for the price that puts
            # the next daily RSI reading at 60
            last_avg_gain = float(daily_avg_gain.iloc[-1])
            last_avg_loss = float(daily_avg_loss.iloc[-1])
            target_rs = TARGET_RSI / (100 - TARGET_RSI)
            price_change_needed = (RSI_PERIOD - 1) * (target_rs * last_avg_loss - last_avg_gain)
            rsi_implied_t1 = cmp + price_change_needed
            # The RSI-implied target can already be behind the current price
            # (price ran up fast after the signal candle) - take the highest of
            # the RSI-implied level, a 2x-risk level from entry, and one more
            # risk unit above cmp, so T1 is always a genuine forward target.
            t1 = max(rsi_implied_t1, entry + (2 * risk), cmp + risk)

            results.append({
                'Date': today_date,
                'Stock': ticker.replace('.TO', ''),
                'CMP': round(cmp, 2),
                'Monthly RSI': round(last_monthly_rsi, 1),
                'Weekly RSI': round(last_weekly_rsi, 1),
                'Signal Date': signal_date.strftime('%d-%m-%Y'),
                'Entry': round(entry, 2),
                'SL': round(sl, 2),
                'T1 (RSI 60 Est.)': round(t1, 2),
                'Action': 'RSI 5-Star Setup',
            })

        except Exception as e:
            pass

    if results:
        df = pd.DataFrame(results)
        df = df.sort_values(by='Weekly RSI', ascending=False)
        return df
    else:
        return pd.DataFrame()

### Run the RSI 5-Star Scanner

In [ ]:
print('\n' + '='*120)
print('RSI 5-Star Scanner')
print('='*120 + '\n')

rsi_result = rsi_five_star_scanner(my_stocks)

### Display RSI 5-Star Results

In [ ]:
if rsi_result.empty:
    print('\nNo stocks matched the RSI 5-Star setup today.\n')
else:
    print(rsi_result.to_string(index=False))
    print('\n' + '='*120)
    print(f'Total stocks: {len(rsi_result)}')
    print('='*120)

### Save RSI 5-Star Results to Excel

In [ ]:
if not rsi_result.empty:
    filename = f'RSI_5Star_Results_{datetime.now().strftime("%d-%m-%Y_%H%M%S")}.xlsx'
    rsi_result.to_excel(filename, index=False)
    print(f'Results saved: {filename}')
else:
    print('No data to save to Excel.')